In [1]:
from datasets import load_dataset
from data_processing import (
    _select_usable_examples,
)
from inter_band_distance import run_inter_band_distance_analysis
import importlib
import gp_internal_diagnostics
importlib.reload(gp_internal_diagnostics)
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
dset_plasticc = load_dataset(
    "MultimodalUniverse/plasticc",
    streaming=True,
    split="train",
).with_format("numpy")


/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_examples, scanned_examples = _select_usable_examples(
    iter(dset_plasticc),
    target_band="r",
    n_objects=500,
    max_examples_to_scan=1000,
    min_points=8,
)

In [3]:
from inter_band_distance import run_inter_band_distance_analysis
rows = []

for example in selected_examples:
    lc = example["lightcurve"]

    time = np.asarray(lc["time"], dtype=float)
    flux = np.asarray(lc["flux"], dtype=float)
    band = np.asarray(lc["band"])

    valid = np.isfinite(time) & np.isfinite(flux)
    time = time[valid]
    flux = flux[valid]
    band = band[valid]

    peak_time = time[np.argmax(np.abs(flux))]

    rows.extend({
        "object_id": example["object_id"],
        "obj_type": example["obj_type"],
        "band": band_i,
        "time_relative_to_peak": time_i - peak_time,
        "flux": flux_i,
    } for time_i, flux_i, band_i in zip(time, flux, band))

observations_df = pd.DataFrame(rows)

distances, correlations = run_inter_band_distance_analysis(
    observations_df,
    output_dir="inter_band_distance_outputs",
)

In [5]:
from inter_band_distance import compute_inter_band_distances
distances, correlations, templates = compute_inter_band_distances(observations_df)
slsn_u = templates[
    (templates["class"] == "SLSN-I") &
    (templates["band"] == "u")
]

print(slsn_u.sort_values("template_flux", key=abs).tail(10))

       class band  time_bin  template_flux
457   SLSN-I    u         0       1.681468
1493  SLSN-I    u        25       2.036094
461   SLSN-I    u        17       2.412723
1490  SLSN-I    u        21       2.628880
2147  SLSN-I    u        26       3.144045
2199  SLSN-I    u        27       4.043059
1491  SLSN-I    u        22       4.552486
463   SLSN-I    u        23       6.708301
1715  SLSN-I    u        20       9.987346
1492  SLSN-I    u        24      82.673052


In [6]:
slsn = observations_df[observations_df["obj_type"] == "SLSN-I"].copy()

scales = slsn.groupby("object_id")["flux"].transform(
    lambda x: np.percentile(np.abs(x), 95)
)
slsn["normalized_flux"] = slsn["flux"] / scales

print(
    slsn.groupby("band")["normalized_flux"]
        .agg(["count", "median", "min", "max"])
)

      count  median       min         max
band                                     
Y      3031     0.0  0.000000    0.000000
g      3031     0.0 -3.208879  331.616638
i      3031     0.0 -2.915697  300.587377
r      3031     0.0 -1.947749  323.999416
u      3031     0.0 -4.542229  250.007847
z      3031     0.0 -5.203073  277.264575


In [8]:
import pandas as pd
# Which observations produce the extreme peak-bin value?
time_bins = np.linspace(-500, 500, 50)

slsn_u = slsn[
    (slsn["band"] == "u") &
    (pd.cut(
        slsn["time_relative_to_peak"],
        time_bins,
        labels=False,
        include_lowest=True,
    ) == 24)
]

print(
    slsn_u[
        ["object_id", "time_relative_to_peak", "flux", "normalized_flux"]
    ].sort_values("normalized_flux", key=abs, ascending=False)
)

       object_id  time_relative_to_peak         flux  normalized_flux
543345     23539               1.988281  1195.256470       250.007847
543346     23539               2.984375  1158.341309       242.286424
543347     23539               3.980469  1136.817017       237.784259
543348     23539               4.980469  1134.926392       237.388803
543349     23539               5.976562  1127.815796       235.901504
543350     23539               6.972656  1096.636108       229.379751
543351     23539               7.964844  1083.784180       226.691556
152619  14888486               0.000000  1526.135864       150.813502
264392  80346495              -9.011719   268.310486        14.532601
64637   22254809              -2.027344    65.312119         4.226781
64639   22254809               2.957031    53.746258         3.478277
64638   22254809               1.964844    50.849777         3.290827


In [7]:
object_scales = (
    slsn.groupby("object_id")["flux"]
        .apply(lambda x: np.percentile(np.abs(x), 95))
        .sort_values()
)

print(object_scales.head(20))

object_id
123493         2.228767
131629         2.665028
80205          3.649259
100164081      4.218725
23539          4.780876
106341515      6.380571
29725628       8.179105
78737520       8.214868
14888486      10.119358
90105327      10.559692
73610         11.192362
22254809      15.451976
80346495      18.462661
21974297      48.892823
42198183     351.588586
Name: flux, dtype: float64
